# Exercises XP: RAG with LangChain (Student)

## 0) Setup


In [ ]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

In [ ]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA


## 1) Load dataset and convert to Documents


In [ ]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

ds = load_dataset(dataset_name, split=split)

print("Colonnes:", ds.column_names)
print("Exemple brut:", ds[0])

documents: List[Document] = []
for row in ds:
    documents.append(
        Document(
            page_content=row[text_column],
            metadata={"source": row[source_column]}
        )
    )

print("\nDocuments chargés:", len(documents))
print("Exemple metadata:", documents[0].metadata)
print("Exemple content:", documents[0].page_content[:350])

## 2) Split into chunks


In [ ]:
chunk_size = 500
chunk_overlap = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
    separators=["\n\n", "\n", ".", " ", ""]
)

splits = splitter.split_documents(documents)
print(f"Nombre de chunks créés: {len(splits)}")
print("Premier chunk metadata:", splits[0].metadata)
print("Premier chunk content:\n", splits[0].page_content[:350])

# Note: Utiliser chunk_size=1000 augmenterait la longueur du contexte mais réduirait le nombre de chunks.

## 3) Vector store + retriever (FAISS)


In [ ]:
from langchain_community.vectorstores import FAISS, DistanceStrategy

embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings,
    distance_strategy=DistanceStrategy.COSINE
)

k = 4 # Modifiable en 2 ou 6 pour tester
retriever = vectorstore.as_retriever(search_kwargs={"k": k})

print("Retriever prêt.")

print("--- Sanity Check ---")
query = "How do I use pipeline?"
docs = retriever.get_relevant_documents(query)
for i, doc in enumerate(docs):
    print(f"\nChunk {i+1} (Source: {doc.metadata['source']}):")
    print(doc.page_content[:200] + "...")

Retriever ready


## 4) Build the RAG chain


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain.chains import RetrievalQA

model_id = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

hf_pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

print("Chaîne RAG prête.")

## 5) Demo: RAG vs no-RAG


In [ ]:
def ask_rag(query):
    result = qa({"query": query})
    print(f"\nQUESTION: {query}")
    print(f"REPONSE: {result['result']}")
    sources = set([doc.metadata['source'] for doc in result['source_documents']])
    print("SOURCES UNIQUES:")
    for s in sources:
        print(f"- {s}")

# Demo
ask_rag("How can I retrieve a model from the Hugging Face Hub?")
ask_rag("What is the use of the tokenizer in transformers?")